# Synthetic primary-care diabetes database

This notebook builds and audits an entirely synthetic SQLite database for a fictional primary-care cohort of 1,200 adults with type 2 diabetes. It is designed to demonstrate relational schema design, reproducible generation, SQL joins, constraints, and transparent data-quality auditing. It is not clinical evidence and must not be used for care decisions.

## Design choices and justifications

- **Cohort and scale:** 1,200 synthetic patients exceed the 1,000-row target. Longitudinal appointments, measurements, diagnoses, medication records, six fictional clinics, and clinicians create related tables with different sizes.
- **Keys and relationships:** stable synthetic identifiers are primary keys where records are events. Bridge/history tables use composite primary keys. Composite foreign keys keep appointment observations linked to the same patient and clinic.
- **Constraints:** `NOT NULL`, `CHECK`, `UNIQUE`, primary keys, foreign keys, and query indexes protect types, allowed values, ranges, and common joins. The observation natural key is intentionally not unique so seeded duplicate measurements remain available for quality analysis.
- **Reproducibility:** the generator uses only Python's standard library and a fixed default seed (`20261007`). Re-running it replaces the selected database file with the same synthetic records.
- **Transparent defects:** 36 missing emails (3.0%), 60 missing smoking statuses (5.0%), 36 missing ethnicity values (3.0%), 12 duplicated contact-email values across 24 people (2.0%), and 36 cloned observation rows are inserted deliberately. These amounts are fixed and audited below.
- **Privacy and ethics:** no source records or real identities are used. Names come from small code-defined lists; clinic labels and contact addresses are fictional, and email addresses use the reserved `.invalid` domain. Small cells and person-level values should not be interpreted as population estimates.

The source file `generate_database.py` is the canonical generation and validation code. Run this notebook from the folder containing that file.


In [1]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys

DB_PATH = Path('synthetic_diabetes.db')
GENERATOR = Path('generate_database.py')
result = subprocess.run(
    [sys.executable, str(GENERATOR), '--db-path', str(DB_PATH), '--seed', '20261007'],
    check=True, capture_output=True, text=True,
)
build_summary = json.loads(result.stdout)
print('Database generated and validated.')
print(json.dumps({
    'seed': build_summary['seed'],
    'integrity_check': build_summary['integrity_check'],
    'foreign_key_violations': build_summary['foreign_key_violations'],
    'constraint_probes_rejected': build_summary['constraint_probes_rejected'],
}, indent=2))

Database generated and validated.
{
  "seed": 20261007,
  "integrity_check": "ok",
  "foreign_key_violations": 0,
  "constraint_probes_rejected": [
    "CHECK",
    "UNIQUE",
    "NOT NULL",
    "FOREIGN KEY",
    "COMPOSITE PRIMARY KEY",
    "MEASUREMENT CHECK"
  ]
}


## 1. Database inventory

The row-count inventory shows the deliberately different scale of lookup, bridge, event, and longitudinal measurement tables.

In [2]:
with sqlite3.connect(DB_PATH) as con:
    row_counts = dict(build_summary['row_counts'])
    for table, count in sorted(row_counts.items()):
        print(f'{table:22s} {count:>7,}')
    assert row_counts['patient'] >= 1000
    assert len(row_counts) >= 8

appointment              6,057
clinic                       6
clinician                   30
clinician_site              30
condition_catalog            6
dataset_metadata             7
medication_catalog           8
observation             13,216
patient                  1,200
patient_condition        3,104
prescription             1,921


## 2. Missing and duplicate data audit

Missing values are represented as SQL `NULL` in nullable fields. Duplicate measurements retain different observation IDs but repeat the same natural event values. The audit reports duplicate groups and extra rows separately so the meaning of the counts is clear.

In [3]:
with sqlite3.connect(DB_PATH) as con:
    con.row_factory = sqlite3.Row
    audit = build_summary['quality_audit']
    for label, count in audit.items():
        print(f'{label:42s} {count:>5,}')
    assert audit['missing_contact_email'] == 36
    assert audit['missing_smoking_status'] == 60
    assert audit['missing_ethnicity'] == 36
    assert audit['duplicate_email_values'] == 12
    assert audit['duplicate_observation_groups'] == 36
    assert audit['extra_duplicate_observation_rows'] == 36

missing_contact_email                         36
missing_smoking_status                        60
missing_ethnicity                             36
duplicate_email_values                        12
patients_in_duplicate_email_groups            24
duplicate_observation_groups                  36
extra_duplicate_observation_rows              36


## 3. Example relational SQL

These queries illustrate joins and aggregation over the synthetic records. They describe the generated dataset only; they are not clinical performance measures.

In [4]:
with sqlite3.connect(DB_PATH) as con:
    rows = con.execute('''
        SELECT c.clinic_name,
               COUNT(a.appointment_id) AS appointments,
               SUM(CASE WHEN a.status = 'attended' THEN 1 ELSE 0 END) AS attended,
               ROUND(100.0 * SUM(CASE WHEN a.status = 'attended' THEN 1 ELSE 0 END)
                     / COUNT(a.appointment_id), 1) AS attended_pct
        FROM clinic AS c
        LEFT JOIN appointment AS a ON a.clinic_id = c.clinic_id
        GROUP BY c.clinic_id, c.clinic_name
        ORDER BY c.clinic_id
    ''').fetchall()
    print('Clinic attendance totals (synthetic events)')
    for row in rows:
        print(f'{row[0]:30s} appointments={row[1]:4d} attended={row[2]:4d} ({row[3]}%)')

Clinic attendance totals (synthetic events)
Northgate Health Centre        appointments=1041 attended= 863 (82.9%)
Riverside Primary Care         appointments=1094 attended= 908 (83.0%)
Meadow Park Surgery            appointments= 998 attended= 824 (82.6%)
Harbour View Clinic            appointments= 901 attended= 737 (81.8%)
Central Lane Practice          appointments=1055 attended= 877 (83.1%)
Hilltop Medical Centre         appointments= 968 attended= 795 (82.1%)


In [5]:
with sqlite3.connect(DB_PATH) as con:
    rows = con.execute('''
        SELECT substr(o.observed_on, 1, 4) AS year,
               COUNT(*) AS hba1c_observations,
               ROUND(AVG(o.value_num), 1) AS mean_mmol_mol
        FROM observation AS o
        WHERE o.metric_code = 'HBA1C'
        GROUP BY substr(o.observed_on, 1, 4)
        ORDER BY year
    ''').fetchall()
    print('Annual synthetic HbA1c records (descriptive only; duplicate rows retained)')
    for row in rows:
        print(f'{row[0]}: n={row[1]:4d}, arithmetic mean={row[2]} mmol/mol')

Annual synthetic HbA1c records (descriptive only; duplicate rows retained)
2023: n=1051, arithmetic mean=59.9 mmol/mol
2024: n=1060, arithmetic mean=59.8 mmol/mol
2025: n=1010, arithmetic mean=59.4 mmol/mol
2026: n= 833, arithmetic mean=59.4 mmol/mol


## 4. Integrity and constraint verification

The generation script runs SQLite integrity and foreign-key checks, then attempts representative invalid inserts inside savepoints. Each probe must be rejected by the intended constraint, and each savepoint is rolled back so the database remains unchanged.

In [6]:
with sqlite3.connect(DB_PATH) as con:
    con.execute('PRAGMA foreign_keys = ON')
    print('integrity_check:', con.execute('PRAGMA integrity_check').fetchone()[0])
    print('foreign_key_check rows:', len(con.execute('PRAGMA foreign_key_check').fetchall()))
    print('rejected constraint probes:', ', '.join(build_summary['constraint_probes_rejected']))
    assert con.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
    assert not con.execute('PRAGMA foreign_key_check').fetchall()
    assert len(build_summary['constraint_probes_rejected']) == 6

integrity_check: ok
foreign_key_check rows: 0
rejected constraint probes: CHECK, UNIQUE, NOT NULL, FOREIGN KEY, COMPOSITE PRIMARY KEY, MEASUREMENT CHECK
